# WeekendPulse Reels — Batch Renderer

Turn today's **reel-worthy** news posts (from `reels_batch.txt`) into short,
~16s vertical Facebook Reels. One reel per story.

## Pipeline (per reel)
1. Ken Burns pan/zoom over the article's **real photo** (branded fallback card if none).
2. **Chatterbox-Nano** TTS narration of the AI's `reel_blurb` — a **random
   female voice**, precise emotion (`neutral` / `excited` / `surprised`).
3. Whisper-aligned **captions** burned in.
4. A **title card** (vibrant orange + white) fades in over the body.
5. Music from the repo `music/` folder starts at position 0 and is **cut at
   narration end** (loopable head, auto-ducked under the voice).
6. A **~0.4s crossfade** into the fixed 4s `outro.mp4`.

## Voices
- **10 female voices** are auto-downloaded from the public
  `OwenTyme/voice-zero` `voices-emotion/` pool (each is a folder of emotion
  clips, e.g. `emily_cripps/excited.flac`).
- Every reel uses a **random female voice**. The AI's `reel_emotion`
  (`neutral` / `excited` / `surprised`) picks that voice's matching clip; if
  missing it falls back to the voice's `neutral.flac`. A voice is never pinned.

## Usage
- **Run all cells in order.** Only Run/Render and Preview are heavy.
- Rendering needs the **T4 GPU** runtime (Chatterbox-Nano). First run downloads
  ~2.9 GB of model weights + 10 female voice clips + whisper model.
- After rendering, use the **Preview** cell to review one reel at a time
  (Next / Previous). Download the ones you like with the **Download** cell.
- **Nothing is auto-pushed to Facebook** — you post manually.

> TIP: shorter is better. Keep each reel ~16s (soft target), anything under
> ~25s is fine. Very long blurb results in a long reel — consider editing the
> blurb in `reels_batch.txt` before you re-render.

In [ ]:
# Cell 2 — Environment. Keep cell order; run all.
import subprocess, sys, os

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, check=True, **kw)

print("Installing dependencies ...")
sh("apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg >/dev/null")
pip_base = [sys.executable, "-m", "pip", "-q", "install"]

# core: TTS runtime + torch extras
sh("-c", " ".join([*pip_base,
                  "chatterbox", "pyloudnorm", "torchaudio",
                  "faster-whisper", "torch",
                  "librosa", "soundfile", "pydub", "Pillow",
                  "ipython", "IPython"]))

print("ffmpeg:", sh("ffmpeg -version | head -n1", capture_output=True, text=True).stdout.strip())
print("Setup done.")


In [ ]:
# Cell — install the renderer modules (reel_render.py, align.py, enhance.py).
import base64
from pathlib import Path
SRC = Path('/content/weekendpulse_reels/story_src')
SRC.mkdir(parents=True, exist_ok=True)
SRC.joinpath('align.py').write_text(
    base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKYWxpZ24ucHkg4oCUIEZvcmNlZC1hbGlnbm1lbnQga2FyYW9rZSBjYXB0aW9uIGdlbmVyYXRvciBmb3IgU2NhcnlUYWxlcyBSZWVscy4KCklucHV0cyAoc2FtZSBiYXNlbmFtZSBpbiBhIGZvbGRlcik6CiAgPG5hbWU+LnR4dCAgIC0+IG5hcnJhdGlvbiB0ZXh0IGlzIGV2ZXJ5dGhpbmcgQkVGT1JFIHRoZSBmaXJzdCAiXG4tLS1cbiIKICAgICAgICAgICAgICAgICAgKHRoZSByZXN0IGlzIHNvY2lhbCBjYXB0aW9uIG1ldGFkYXRhIHdlIGlnbm9yZSkKICA8bmFtZT4ud2F2ICAgLT4gdGhlIG5hcnJhdGlvbiBhdWRpbyAob3IgLm1wMykKT3V0cHV0czoKICA8bmFtZT4uYXNzICAgLT4ga2FyYW9rZSAod29yZC1yZXZlYWwpIEFkdmFuY2VkIFN1YlN0YXRpb24gQWxwaGEgc3VidGl0bGVzCiAgICAgICAgICAgICAgICAgIGJ1cm5lZCB3aXRoIEZGbXBlZzogIGZmbXBlZyAtaSBpbWcgLWkgYXVkaW8gLXZmIGFzcz08bmFtZT4uYXNzCgpUaGUgb24tc2NyZWVuIHdvcmRzIGFsd2F5cyBtYXRjaCB0aGUgV1JJVFRFTiBuYXJyYXRpb24sIHdoaWxlIHRpbWluZyBjb21lcwpmcm9tIGZhc3Rlci13aGlzcGVyJ3MgZGV0ZWN0ZWQgc3BlZWNoIChmb3JjZWQgYWxpZ25tZW50IG9mIHRoZSBrbm93biB0ZXh0Cm9udG8gdGhlIGF1ZGlvIHRpbWVsaW5lKS4gV29yZC1ieS13b3JkIGthcmFva2UgcmV2ZWFsLgoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzeXMKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB0ZXh0IHV0aWxzCgpQVU5DVCA9ICIuLCE/OzpcIifigJzigJ3igJjigJkoKVtdLeKAk+KAlOKApiIKCgpkZWYgbm9ybWFsaXplX3dvcmQodzogc3RyKSAtPiBzdHI6CiAgICAiIiJMb3dlcmNhc2UsIHN0cmlwIHB1bmN0dWF0aW9uLCBjb2xsYXBzZSBhcG9zdHJvcGhlcy9zaHkgdmFyaWF0aW9uLiIiIgogICAgdyA9IHcubG93ZXIoKQogICAgdyA9IHJlLnN1YihyIltcdTIwMThcdTIwMTlcdTAwMjddIiwgIiciLCB3KSAgICMgY3VybHktPnN0cmFpZ2h0IGFwb3N0cm9waGUKICAgIHcgPSB3LnJlcGxhY2UoIuKAmSIsICInIikKICAgICMgc3RyaXAgZXZlcnl0aGluZyBleGNlcHQgbGV0dGVycywgZGlnaXRzLCBhcG9zdHJvcGhlLCBoeXBoZW4KICAgIHcgPSByZS5zdWIociJbXmEtejAtOSdcLV0rIiwgIiIsIHcpCiAgICByZXR1cm4gdwoKCmRlZiB0b2tlbml6ZSh0ZXh0OiBzdHIpOgogICAgIiIiUmV0dXJuIGxpc3Qgb2Ygd29yZHMgd2l0aCBhcHByb3hpbWF0ZSBjaGFyIHBvc2l0aW9ucyAoZm9yIHB1bmN0dWF0aW9uCiAgICByZWNvdmVyeSksIHNwbGl0dGluZyBvbiB3aGl0ZXNwYWNlIGJ1dCBrZWVwaW5nIHB1bmN0dWF0aW9uIGF0dGFjaGVkLiIiIgogICAgIyBXZSBzcGxpdCBvbiB3aGl0ZXNwYWNlOyBwdW5jdHVhdGlvbiBzdGF5cyBhdHRhY2hlZCB0byB0b2tlbnMuCiAgICAjIENoYXIgc3BhbnMgYXJlIG9ubHkgdXNlZCB0byByZWNvdmVyIHRoZSBPUklHSU5BTCB0b2tlbiBzdWJzdHJpbmcuCiAgICB0b2tlbnMgPSBbXQogICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIociJcUysiLCB0ZXh0KToKICAgICAgICB0b2tlbnMuYXBwZW5kKHsicmF3IjogbS5ncm91cCgwKSwgInN0YXJ0IjogbS5zdGFydCgpLCAiZW5kIjogbS5lbmQoKX0pCiAgICByZXR1cm4gdG9rZW5zCgoKZGVmIHBhcnNlX2V2ZW50cyhyYXcpOgogICAgIiIicmF3OiBsaXN0IG9mIHNlZ21lbnQgZGljdHMgZnJvbSBmYXN0ZXItd2hpc3BlciB3aXRoIC53b3Jkcy4KICAgIFJldHVybnMgbGlzdCBvZiB7IndvcmQiOi4uLiwgInN0YXJ0IjouLi4sICJlbmQiOi4uLn0gZmxhdHRlbmVkIGluIG9yZGVyLiIiIgogICAgZXZlbnRzID0gW10KICAgIGZvciBzZWcgaW4gcmF3OgogICAgICAgIGZvciB3IGluIHNlZy5nZXQoIndvcmRzIiwgW10pOgogICAgICAgICAgICBzdGFydCA9IHcuZ2V0KCJzdGFydCIpCiAgICAgICAgICAgIGVuZCA9IHcuZ2V0KCJlbmQiKQogICAgICAgICAgICBpZiBzdGFydCBpcyBOb25lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZXZlbnRzLmFwcGVuZCh7IndvcmQiOiBub3JtYWxpemVfd29yZCh3LmdldCgid29yZCIsICIiKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJzdGFydCI6IGZsb2F0KHN0YXJ0KSwgImVuZCI6IGZsb2F0KGVuZCl9KQogICAgcmV0dXJuIGV2ZW50cwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBhbGlnbm1lbnQgY29yZQoKZGVmIGFsaWduKG5hcnJhdGlvbl9ldmVudHMsIHdoaXNwZXJfZXZlbnRzKToKICAgICIiIkdyZWVkeSBsZWZ0LXRvLXJpZ2h0IGFsaWdubWVudC4KCiAgICBuYXJyYXRpb25fZXZlbnRzIDogb3JkZXJlZCB3cml0dGVuIHdvcmRzIChub3JtYWxpemVkKSB3ZSB3YW50IHRpbWluZ3MgZm9yCiAgICB3aGlzcGVyX2V2ZW50cyAgICA6IG9yZGVyZWQgZGV0ZWN0ZWQgKG5vcm1hbGl6ZWQpIHdvcmRzIHdpdGggdGltaW5ncwoKICAgIFJldHVybnM6IGxpc3Qgb2YgZGljdHMgZm9yIHRoZSBXUklUVEVOIHdvcmRzOgogICAgICAgIHsid29yZCI6IG9yaWdpbmFsIHdyaXR0ZW4gZGlzcGxheSB0b2tlbiwgInN0YXJ0IjouLiwgImVuZCI6Li59CiAgICBBIHdvcmQgdGhhdCBjb3VsZCBub3QgYmUgbWF0Y2hlZCBmYWxscyBiYWNrIHRvIHRoZSB0aW1lIG9mIGl0cyBuZWFyZXN0CiAgICBtYXRjaGVkIG5laWdoYm91ciAoY2xhbXBlZCkgc28gdGhlIHRpbWVsaW5lIG5ldmVyIGdhcHMuCiAgICAiIiIKICAgICMgQnVpbGQgc2VxdWVuY2Ugb2Ygd2hpc3BlciBub3JtYWxpemVkIHdvcmRzIHRvIGFsbG93IHNraXBwaW5nIG5vaXNlCiAgICBuID0gbGVuKG5hcnJhdGlvbl9ldmVudHMpCiAgICAjIFdlJ2xsIHdhbGsgd2hpc3BlciBpbmRleCBmb3J3YXJkLCBtYXRjaGluZyBhcyBtYW55IG5hcnJhdGlvbiB3b3JkcyBhcwogICAgIyBwb3NzaWJsZS4gRm9yIHVubWF0Y2hlZCB3aGlzcGVyIHRva2VucyB3ZSBqdXN0IHNraXAgdGhlbS4KICAgIHJlc3VsdCA9IFtOb25lXSAqIG4KICAgIHdpID0gMAogICAgd25fdG90YWwgPSBsZW4od2hpc3Blcl9ldmVudHMpCgogICAgIyBGaXJzdCBwYXNzOiBtYXRjaCBlYWNoIG5hcnJhdGlvbiB3b3JkIHRvIGEgd2hpc3BlciB3b3JkIHRpbWluZy4KICAgICMgd2hpc3BlciB3b3JkIGkgY29ycmVzcG9uZHMgdG8gbmFycmF0aW9uIHdvcmQgaSBpbiBhIDE6MSBjbGVhbiByZWFkLCBidXQKICAgICMgd2hpc3BlciBtYXkgZHJvcC9yZW9yZGVyOyB1c2UgYSBtb3Zpbmcgd2luZG93IHNlYXJjaCBmb3IgYSBtYXRjaC4KICAgIGkgPSAwCiAgICB3aGlsZSBpIDwgbjoKICAgICAgICB0YXJnZXQgPSBuYXJyYXRpb25fZXZlbnRzW2ldWyJub3JtIl0KICAgICAgICBmb3VuZCA9IE5vbmUKICAgICAgICAjIHNlYXJjaCBmb3J3YXJkIGluIHdoaXNwZXIgZXZlbnRzIGJ5IHVwIHRvIGEgZmV3IHRva2VucwogICAgICAgIHNlYXJjaF9saW1pdCA9IG1pbih3bl90b3RhbCwgd2kgKyA2KQogICAgICAgIGZvciBqIGluIHJhbmdlKHdpLCBzZWFyY2hfbGltaXQpOgogICAgICAgICAgICBpZiB3aGlzcGVyX2V2ZW50c1tqXVsid29yZCJdID09IHRhcmdldDoKICAgICAgICAgICAgICAgIGZvdW5kID0gagogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBmb3VuZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmVzdWx0W2ldID0gKHdoaXNwZXJfZXZlbnRzW2ZvdW5kXVsic3RhcnQiXSwKICAgICAgICAgICAgICAgICAgICAgICAgIHdoaXNwZXJfZXZlbnRzW2ZvdW5kXVsiZW5kIl0pCiAgICAgICAgICAgIHdpID0gZm91bmQgKyAxCiAgICAgICAgICAgIGkgKz0gMQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgbmFycmF0aW9uIHdvcmQgbm90IGZvdW5kIC0+IG1hcmsgbWlzc2luZzsga2VlcCB3aSwgYWR2YW5jZSBpCiAgICAgICAgICAgIHJlc3VsdFtpXSA9IE5vbmUKICAgICAgICAgICAgaSArPSAxCgogICAgIyBTZWNvbmQgcGFzczogZmlsbCBtaXNzaW5nIHRpbWluZ3MgYnkgaW50ZXJwb2xhdGlvbiBiZXR3ZWVuIGtub3duIGFuY2hvcnMuCiAgICBrbm93bl9pZHggPSBbayBmb3IgaywgdiBpbiBlbnVtZXJhdGUocmVzdWx0KSBpZiB2IGlzIG5vdCBOb25lXQogICAgaWYgbm90IGtub3duX2lkeDoKICAgICAgICAjIG5vdGhpbmcgbWF0Y2hlZCBhdCBhbGwgLT4gZ2l2ZSBlYWNoIHdvcmQgYSBmbGF0IHNoYXJlIG9mIGF1ZGlvIDAuLlgKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIHdvcmRzIG1hdGNoZWQgYmV0d2VlbiBuYXJyYXRpb24gYW5kIGF1ZGlvICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIi0gY2hlY2sgdGhlIG5hcnJhdGlvbiB0ZXh0IG1hdGNoZXMgdGhlIGF1ZGlvLiIpCiAgICAjIEZvciBlYWNoIG1pc3NpbmcgaW5kZXggYmV0d2VlbiBrbm93biBhbmNob3JzLCBsaW5lYXIgaW50ZXJwb2xhdGUuCiAgICBmb3IgaSBpbiByYW5nZShuKToKICAgICAgICBpZiByZXN1bHRbaV0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIyBmaW5kIHByZXYga25vd24KICAgICAgICBwcmV2cyA9IFtrIGZvciBrIGluIGtub3duX2lkeCBpZiBrIDwgaV0KICAgICAgICBuZXh0cyA9IFtrIGZvciBrIGluIGtub3duX2lkeCBpZiBrID4gaV0KICAgICAgICBpZiBwcmV2cyBhbmQgbmV4dHM6CiAgICAgICAgICAgIHAgPSBwcmV2c1stMV0KICAgICAgICAgICAgbnggPSBuZXh0c1swXQogICAgICAgICAgICBmcmFjID0gKGkgLSBwKSAvIChueCAtIHApCiAgICAgICAgICAgIHMgPSByZXN1bHRbcF1bMF0gKyAocmVzdWx0W254XVswXSAtIHJlc3VsdFtwXVswXSkgKiBmcmFjCiAgICAgICAgICAgIGUgPSByZXN1bHRbcF1bMV0gKyAocmVzdWx0W254XVsxXSAtIHJlc3VsdFtwXVsxXSkgKiBmcmFjCiAgICAgICAgZWxpZiBwcmV2czoKICAgICAgICAgICAgcCA9IHByZXZzWy0xXQogICAgICAgICAgICBkdXIgPSAocmVzdWx0W3BdWzFdIC0gcmVzdWx0W3BdWzBdKSBvciAwLjE1CiAgICAgICAgICAgIHMgPSByZXN1bHRbcF1bMV0KICAgICAgICAgICAgZSA9IHMgKyBkdXIKICAgICAgICBlbGlmIG5leHRzOgogICAgICAgICAgICBueCA9IG5leHRzWzBdCiAgICAgICAgICAgIGR1ciA9IChyZXN1bHRbbnhdWzFdIC0gcmVzdWx0W254XVswXSkgb3IgMC4xNQogICAgICAgICAgICBlID0gcmVzdWx0W254XVswXQogICAgICAgICAgICBzID0gZSAtIGR1cgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcmVzdWx0W2ldID0gKHMsIGUpCgogICAgIyBCdWlsZCBvdXRwdXQsIG9uZSBlbnRyeSBwZXIgd3JpdHRlbiBkaXNwbGF5IHRva2VuLgogICAgb3V0ID0gW10KICAgIGZvciBpLCBldiBpbiBlbnVtZXJhdGUobmFycmF0aW9uX2V2ZW50cyk6CiAgICAgICAgcywgZSA9IHJlc3VsdFtpXQogICAgICAgIG91dC5hcHBlbmQoeyJ3b3JkIjogZXZbInJhdyJdLCAic3RhcnQiOiByb3VuZChzLCAzKSwgImVuZCI6IHJvdW5kKGUsIDMpfSkKICAgIHJldHVybiBvdXQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gQVNTIG91dHB1dAoKZGVmIHRpbWVzdGFtcF9hc3ModCk6CiAgICAiIiJBU1MgdGltZSBmb3JtYXQgSDpNTTpTUy5jYyAoY2VudGlzZWNvbmRzKS4iIiIKICAgIHQgPSBtYXgoMC4wLCB0KQogICAgaCA9IGludCh0IC8vIDM2MDApCiAgICBtID0gaW50KCh0ICUgMzYwMCkgLy8gNjApCiAgICBzID0gaW50KHQgJSA2MCkKICAgIGNzID0gaW50KHJvdW5kKCh0IC0gaW50KHQpKSAqIDEwMCkpCiAgICByZXR1cm4gZiJ7aH06e206MDJkfTp7czowMmR9LntjczowMmR9IgoKCmRlZiBhc3NfaGVhZGVyKGZvbnRzaXplPTcwLCBmb250bmFtZT0iQ2hpbGxlciIsIGFsaWdubWVudD01LAogICAgICAgICAgICAgICBvdXRsaW5lPTYsIHNoYWRvdz0zLCBvdXRsaW5lX2NvbG91cj0iJkgwMDIyMjIyMiIsCiAgICAgICAgICAgICAgIGJhY2tfY29sb3VyPSImSDgwMDAwMDAwIik6CiAgICAiIiJCdWlsZCB0aGUgW1NjcmlwdCBJbmZvXSArIFtWNCsgU3R5bGVzXSBibG9jay4KCiAgICBEZWZhdWx0IGFsaWdubWVudD01IC0+IHRoZSBjYXB0aW9uIGJsb2NrIGlzIGNlbnRyZWQgYm90aCBob3Jpem9udGFsbHkgYW5kCiAgICB2ZXJ0aWNhbGx5IChtaWRkbGUgb2YgdGhlIGZyYW1lKSwgTk9UIHBpbm5lZCB0byB0aGUgYm90dG9tLgogICAgIiIiCiAgICByZXR1cm4gKAogICAgICAgIGYiW1NjcmlwdCBJbmZvXVxuIgogICAgICAgIGYiU2NyaXB0VHlwZTogdjQuMDArXG4iCiAgICAgICAgZiJQbGF5UmVzWDogMTA4MFxuIgogICAgICAgIGYiUGxheVJlc1k6IDE5MjBcbiIKICAgICAgICBmIldyYXBTdHlsZTogMFxuIgogICAgICAgIGYiU2NhbGVkQm9yZGVyQW5kU2hhZG93OiB5ZXNcbiIKICAgICAgICBmIlxuIgogICAgICAgIGYiW1Y0KyBTdHlsZXNdXG4iCiAgICAgICAgZiJGb3JtYXQ6IE5hbWUsIEZvbnRuYW1lLCBGb250c2l6ZSwgUHJpbWFyeUNvbG91ciwgU2Vjb25kYXJ5Q29sb3VyLCBPdXRsaW5lQ29sb3VyLCBCYWNrQ29sb3VyLCBCb2xkLCBJdGFsaWMsIFVuZGVybGluZSwgU3RyaWtlT3V0LCBTY2FsZVgsIFNjYWxlWSwgU3BhY2luZywgQW5nbGUsIEJvcmRlclN0eWxlLCBPdXRsaW5lLCBTaGFkb3csIEFsaWdubWVudCwgTWFyZ2luTCwgTWFyZ2luUiwgTWFyZ2luViwgRW5jb2RpbmdcbiIKICAgICAgICBmIlN0eWxlOiBTdWIse2ZvbnRuYW1lfSx7Zm9udHNpemV9LCZIMDBGRkZGRkYsJkgwMDU1NTU1NSwiCiAgICAgICAgZiJ7b3V0bGluZV9jb2xvdXJ9LHtiYWNrX2NvbG91cn0sLTEsMCwwLDAsMTAwLDEwMCwwLDAsMSwiCiAgICAgICAgZiJ7b3V0bGluZX0se3NoYWRvd30se2FsaWdubWVudH0sNjAsNjAsMCwxXG4iCiAgICAgICAgZiJcbiIKICAgICAgICBmIltFdmVudHNdXG4iCiAgICAgICAgZiJGb3JtYXQ6IExheWVyLCBTdGFydCwgRW5kLCBTdHlsZSwgTmFtZSwgTWFyZ2luTCwgTWFyZ2luUiwgTWFyZ2luViwgRWZmZWN0LCBUZXh0XG4iCiAgICApCgoKIyBQdW5jdHVhdGlvbiB0aGF0IGlzIGFsd2F5cyBzdHJpcHBlZCBmcm9tIGluc2lkZSBhIG1lcmdlZCBjYXB0aW9uIGFuZCBvbmx5CiMgcmUtYXBwZW5kZWQgKGluIG9yZGVyKSBhdCB0aGUgdmVyeSBlbmQgb2YgdGhlIGNhcHRpb24uIEhhbmRsZXMgdGhlIHN0YW5kYXJkCiMgc2V0IHBsdXMgY3VybHkgcXVvdGVzIC8gYXBvc3Ryb3BoZXMgLyBoeXBoZW5zIC8gZGFzaGVzLgpTVFJJUF9SRSA9IHJlLmNvbXBpbGUociJbLiw7OiE/XCInXHUyMDE4XHUyMDE5XHUyMDFjXHUyMDFkKClcW1xdXHUyMDEzXHUyMDE0XC1dKyIpCgoKZGVmIGJ1aWxkX2FzcyhhbGlnbmVkLCB3aWR0aD0xMDgwLCBoZWlnaHQ9MTkyMCwgbWF4X2NoYXJzX3Blcl9saW5lPTMwLAogICAgICAgICAgICAgIGZvbnRzaXplPTY0LCBmb250bmFtZT0iQ2hpbGxlciIsIG1pbl9ob2xkPTAuNDUpOgogICAgIiIiYWxpZ25lZDogbGlzdCBvZiB7d29yZCwgc3RhcnQsIGVuZH0uIFByb2R1Y2UgdmVydGljYWxseS1jZW50cmVkIHN1YnRpdGxlCiAgICBldmVudHMgKE5PVCBiYWNrc2xhc2gtayBrYXJhb2tlLCB3aGljaCBsaWJhc3MgaW4gZmZtcGVnIHJlbmRlcnMgYXMgbGl0ZXJhbAogICAgdGV4dCkuCgogICAgTWVyZ2luZyBydWxlczoKICAgICAgKiB3b3JkcyBhcmUgZ3JvdXBlZCBpbnRvIHJlYWRhYmxlIGNodW5rcyBzbyBhIGNodW5rIHN0YXlzIHZpc2libGUKICAgICAgICA+PSBtaW5faG9sZCBzZWNvbmRzIChmaXhlcyBmYXN0LXNwZWVjaCBzaW5nbGUtd29yZCBmbGlja2VyKQogICAgICAqIGEgZ3JvdXAgTkVWRVIgY3Jvc3NlcyBhIHNlbnRlbmNlIGJvdW5kYXJ5IChhIHdvcmQgZW5kaW5nIGluIC4/ISApLAogICAgICAgIHNvIHRoZSBlbmQgb2Ygb25lIHNlbnRlbmNlIG5ldmVyIG1lcmdlcyB3aXRoIHRoZSBzdGFydCBvZiBhbm90aGVyCiAgICAgICogZ3JvdXBzIGFyZSBlbWl0dGVkIHdpdGggbm8gdGltZSBvdmVybGFwIChlYWNoIGhhcyBzdGFydCA+PSBwcmV2IGVuZCksCiAgICAgICAgZXZlbiB3aGVuIHdoaXNwZXIncyBvd24gd29yZCB0aW1pbmdzIG92ZXJsYXAKICAgICAgKiBOTyBwdW5jdHVhdGlvbiBldmVyIGFwcGVhcnMgaW5zaWRlIGEgbWVyZ2VkIGNhcHRpb246IGV2ZXJ5ICwgOyA6ICEgPwogICAgICAgICIgJyAoICkgLSAuLi4gaXMgc3RyaXBwZWQgZnJvbSB0aGUgd29yZHMsIGFuZCBvbmx5IHRoZSBwdW5jdHVhdGlvbiB0aGF0CiAgICAgICAgdHJhaWxlZCB0aGUgRklOQUwgd29yZCBvZiB0aGUgZ3JvdXAgaXMgcmUtYXBwZW5kZWQgYXQgdGhlIHZlcnkgZW5kIG9mCiAgICAgICAgdGhlIGNhcHRpb24gbGluZS4KICAgICIiIgogICAgZXZlbnRzID0gW10KCiAgICAjIC0tLS0gZ3JvdXBpbmcgLS0tLQogICAgZ3JvdXBzID0gW10KICAgIGN1cl93b3JkcyA9IFtdCiAgICBkZWYgY2xvc2UoKToKICAgICAgICBub25sb2NhbCBjdXJfd29yZHMKICAgICAgICBpZiBjdXJfd29yZHM6CiAgICAgICAgICAgIGdyb3Vwcy5hcHBlbmQoY3VyX3dvcmRzKQogICAgICAgICAgICBjdXJfd29yZHMgPSBbXQoKICAgIGZvciBpdGVtIGluIGFsaWduZWQ6CiAgICAgICAgaWYgbm90IGN1cl93b3JkczoKICAgICAgICAgICAgY3VyX3dvcmRzID0gW2l0ZW1dCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY3VyX3dvcmRzLmFwcGVuZChpdGVtKQogICAgICAgICAgICBzcGFuID0gaXRlbVsiZW5kIl0gLSBjdXJfd29yZHNbMF1bInN0YXJ0Il0KICAgICAgICAgICAgaWYgc3BhbiA+PSBtaW5faG9sZDoKICAgICAgICAgICAgICAgIGNsb3NlKCkKICAgICAgICAjIEFsd2F5cyBicmVhayB0aGUgZ3JvdXAgaWYgdGhpcyB3b3JkIGVuZHMgYSBzZW50ZW5jZSwgc28gdGhlIG5leHQKICAgICAgICAjIHNlbnRlbmNlIHN0YXJ0cyBpdHMgb3duIGNhcHRpb24uCiAgICAgICAgaWYgaXRlbVsid29yZCJdLnJzdHJpcCgpLmVuZHN3aXRoKCgiLiIsICIhIiwgIj8iKSk6CiAgICAgICAgICAgIGNsb3NlKCkKICAgIGNsb3NlKCkKCiAgICAjIC0tLS0gZW1pdCB3aXRoIGEgZ3VhcmFudGVlZCBub24tb3ZlcmxhcHBpbmcgdGltZWxpbmUgLS0tLQogICAgcHJldl9lbmQgPSBOb25lCiAgICBmb3Igd29yZHMgaW4gZ3JvdXBzOgogICAgICAgIHN0YXJ0ID0gd29yZHNbMF1bInN0YXJ0Il0KICAgICAgICBlbmQgPSBtYXgod1siZW5kIl0gZm9yIHcgaW4gd29yZHMpCgogICAgICAgICMgTm8tb3ZlcmxhcCBndWFyYW50ZWU6IHRoaXMgY2FwdGlvbiBtYXkgbm90IGJlZ2luIGJlZm9yZSB0aGUgcHJldmlvdXMKICAgICAgICAjIG9uZSBlbmRlZCAod2hpc3BlciB3b3JkIHRpbWluZ3MgY2FuIG92ZXJsYXAgZm9yIGZhc3Qgc3BlZWNoKS4KICAgICAgICBpZiBwcmV2X2VuZCBpcyBub3QgTm9uZSBhbmQgc3RhcnQgPCBwcmV2X2VuZDoKICAgICAgICAgICAgc3RhcnQgPSBwcmV2X2VuZAogICAgICAgIGlmIGVuZCA8PSBzdGFydDoKICAgICAgICAgICAgZW5kID0gc3RhcnQgKyAwLjMKICAgICAgICAjIEVuc3VyZSBhIG1pbmltdW0gdmlzaWJsZSB3aW5kb3cgZXZlbiBmb3IgYSBsb25lIGxvbmcgdGFpbC4KICAgICAgICBpZiBlbmQgLSBzdGFydCA8IDAuMzoKICAgICAgICAgICAgZW5kID0gc3RhcnQgKyAwLjMKICAgICAgICBwcmV2X2VuZCA9IGVuZAoKICAgICAgICAjIFN0cmlwIEFMTCBwdW5jdHVhdGlvbiBmcm9tIGV2ZXJ5IHdvcmQuIFRoZSBvbmx5IHB1bmN0dWF0aW9uIGtlcHQgaXMKICAgICAgICAjIHdoYXRldmVyIHRyYWlsZWQgdGhlIEZJTkFMIHdvcmQgb2YgdGhlIGdyb3VwLCBhcHBlbmRlZCBhdCB0aGUgdmVyeQogICAgICAgICMgZW5kIG9mIHRoZSBjYXB0aW9uIOKAlCBzbyB0aGUgY2FwdGlvbiBuZXZlciBjYXJyaWVzICwgOyAuICEgPyBtaWQtdGV4dC4KICAgICAgICBsYXN0ID0gd29yZHNbLTFdWyJ3b3JkIl0KICAgICAgICBsYXN0X3RyYWlsaW5nID0gIiIuam9pbigKICAgICAgICAgICAgY2ggZm9yIGNoIGluIGxhc3QKICAgICAgICAgICAgaWYgY2ggaW4gIi4sOzohP1wiJ1x1MjAxOFx1MjAxOVx1MjAxY1x1MjAxZCgpW11cdTIwMTNcdTIwMTQtIgogICAgICAgICkKICAgICAgICBjbGVhbl93b3JkcyA9IFtTVFJJUF9SRS5zdWIoIiIsIHdbIndvcmQiXSkgZm9yIHcgaW4gd29yZHNdCiAgICAgICAgY2xlYW5fd29yZHMgPSBbYyBmb3IgYyBpbiBjbGVhbl93b3JkcyBpZiBjXQoKICAgICAgICAjIFdyYXAgbG9uZyBncm91cHMgaW50byBtdWx0aXBsZSBcTi1zZXBhcmF0ZWQgbGluZXMuCiAgICAgICAgdGV4dF9wYXJ0cyA9IFtdCiAgICAgICAgbGluZV9jaGFycyA9IDAKICAgICAgICBsaW5lID0gW10KICAgICAgICBmb3IgY3cgaW4gY2xlYW5fd29yZHM6CiAgICAgICAgICAgIHdsZW4gPSBsZW4oY3cpICsgMSAgIyArMSBmb3IgYSBzcGFjZSBiZXR3ZWVuIHdvcmRzCiAgICAgICAgICAgIGlmIGxpbmUgYW5kIGxpbmVfY2hhcnMgKyB3bGVuID4gbWF4X2NoYXJzX3Blcl9saW5lOgogICAgICAgICAgICAgICAgdGV4dF9wYXJ0cy5hcHBlbmQoIiAiLmpvaW4obGluZSkpCiAgICAgICAgICAgICAgICBsaW5lID0gW2N3XQogICAgICAgICAgICAgICAgbGluZV9jaGFycyA9IGxlbihjdykKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGxpbmUuYXBwZW5kKGN3KQogICAgICAgICAgICAgICAgbGluZV9jaGFycyArPSB3bGVuCiAgICAgICAgaWYgbGluZToKICAgICAgICAgICAgdGV4dF9wYXJ0cy5hcHBlbmQoIiAiLmpvaW4obGluZSkpCiAgICAgICAgY2FwdGlvbiA9ICJcXE4iLmpvaW4odGV4dF9wYXJ0cykKICAgICAgICAjIEFwcGVuZCB0aGUgZmluYWwgd29yZCdzIHRyYWlsaW5nIHB1bmN0dWF0aW9uIGF0IHRoZSB2ZXJ5IGVuZC4KICAgICAgICBpZiBsYXN0X3RyYWlsaW5nOgogICAgICAgICAgICBjYXB0aW9uID0gY2FwdGlvbiArIGxhc3RfdHJhaWxpbmcKCiAgICAgICAgZXZlbnRzLmFwcGVuZCgKICAgICAgICAgICAgZiJEaWFsb2d1ZTogMCx7dGltZXN0YW1wX2FzcyhzdGFydCl9LHt0aW1lc3RhbXBfYXNzKGVuZCl9LCIKICAgICAgICAgICAgZiJTdWIsLDAsMCwwLCx7Y2FwdGlvbn0iCiAgICAgICAgKQoKICAgIHJldHVybiBhc3NfaGVhZGVyKGZvbnRzaXplPWZvbnRzaXplLCBmb250bmFtZT1mb250bmFtZSkgKyAiXG4iLmpvaW4oZXZlbnRzKSArICJcbiIKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1haW4KCmRlZiBhc3Jfd29yZHMoYXVkaW9fcGF0aCk6CiAgICAiIiJSdW4gZmFzdGVyLXdoaXNwZXIsIHJldHVybiBmbGF0dGVuZWQgd29yZCBldmVudHMgKyByYXcgc2VnbWVudHMuIiIiCiAgICBmcm9tIGZhc3Rlcl93aGlzcGVyIGltcG9ydCBXaGlzcGVyTW9kZWwKICAgIG1vZGVsID0gV2hpc3Blck1vZGVsKCJiYXNlIiwgZGV2aWNlPSJjcHUiLCBjb21wdXRlX3R5cGU9ImludDgiKQogICAgc2VnbWVudHMsIF9pbmZvID0gbW9kZWwudHJhbnNjcmliZSgKICAgICAgICBhdWRpb19wYXRoLCB3b3JkX3RpbWVzdGFtcHM9VHJ1ZSwgbGFuZ3VhZ2U9ImVuIiwKICAgICAgICBpbml0aWFsX3Byb21wdD0iSG9ycm9yIHN0b3J5IG5hcnJhdGlvbiBpbiBFbmdsaXNoLiIKICAgICkKICAgIHJhdyA9IFtdCiAgICBmb3Igc2VnIGluIHNlZ21lbnRzOgogICAgICAgIHdvcmRzID0gc2VnLndvcmRzIG9yIFtdCiAgICAgICAgcmF3LmFwcGVuZCh7InRleHQiOiBzZWcudGV4dCwgInN0YXJ0Ijogc2VnLnN0YXJ0LCAiZW5kIjogc2VnLmVuZCwKICAgICAgICAgICAgICAgICAgICAid29yZHMiOiBbCiAgICAgICAgICAgICAgICAgICAgICAgIHsid29yZCI6IHcud29yZCwgInN0YXJ0Ijogdy5zdGFydCwgImVuZCI6IHcuZW5kfQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgdyBpbiB3b3JkcwogICAgICAgICAgICAgICAgICAgIF19KQogICAgcmV0dXJuIHBhcnNlX2V2ZW50cyhyYXcpCgoKZGVmIGdldF9uYXJyYXRpb25fdGV4dCh0eHRfcGF0aCk6CiAgICAiIiJFdmVyeXRoaW5nIGJlZm9yZSB0aGUgZmlyc3QgJy0tLScgbGluZSBpbiB0aGUgc3RvcnkgZmlsZS4iIiIKICAgIHdpdGggb3Blbih0eHRfcGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBjb250ZW50ID0gZi5yZWFkKCkKICAgIHBhcnQgPSBjb250ZW50LnNwbGl0KCJcbi0tLSIsIDEpWzBdLnN0cmlwKCkKICAgIGlmIG5vdCBwYXJ0OgogICAgICAgICMgZmFsbCBiYWNrOiB3aG9sZSBmaWxlCiAgICAgICAgcGFydCA9IGNvbnRlbnQuc3RyaXAoKQogICAgcmV0dXJuIHBhcnQKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iRm9yY2VkLWFsaWduIGNhcHRpb25zIHRvIGF1ZGlvIikKICAgIGFwLmFkZF9hcmd1bWVudCgibmFtZSIsIGhlbHA9ImJhc2VuYW1lIChiYWJ5c2l0dGVyKSAtIGxvb2tzIGZvciBcCiAgICAgICAgPG5hbWU+LnR4dCwgPG5hbWU+LndhdnwubXAzIGluIENXRCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZm9udHNpemUiLCB0eXBlPWludCwgZGVmYXVsdD02NCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1mb250bmFtZSIsIGRlZmF1bHQ9IkNoaWxsZXIiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImZvbnQgZmFtaWx5IGZvciB0aGUgY2FwdGlvbnMgKGUuZy4gQ2hpbGxlciwgQXJpYWwpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXhjaGFycyIsIHR5cGU9aW50LCBkZWZhdWx0PTMwLAogICAgICAgICAgICAgICAgICAgIGhlbHA9Im1heCBjaGFycyBwZXIgY2FwdGlvbiBsaW5lIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1taW4taG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC40NSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJzZWNvbmRzIGVhY2ggY2FwdGlvbiBzdGF5cyB2aXNpYmxlOyBsYXJnZXIgPSBtb3JlICIKICAgICAgICAgICAgICAgICAgICAgICAgICJ3b3JkcyBtZXJnZWQgaW50byByZWFkYWJsZSBjaHVua3MgKGRlZmF1bHQgMC40NSkiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGJhc2UgPSBhcmdzLm5hbWUKICAgIGhlcmUgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKICAgIHR4dCA9IG9zLnBhdGguam9pbihoZXJlLCBiYXNlICsgIi50eHQiKQogICAgYXVkaW8gPSBOb25lCiAgICBmb3IgZXh0IGluICgiLndhdiIsICIubXAzIiwgIi5tNGEiLCAiLmZsYWMiKToKICAgICAgICBwID0gb3MucGF0aC5qb2luKGhlcmUsIGJhc2UgKyBleHQpCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMocCk6CiAgICAgICAgICAgIGF1ZGlvID0gcAogICAgICAgICAgICBicmVhawogICAgaWYgYXVkaW8gaXMgTm9uZToKICAgICAgICBwcmludCgiRVJST1I6IG5vIGF1ZGlvICg8bmFtZT4ud2F2Ly5tcDMvLm00YS8uZmxhYykgZm91bmQgbmV4dCB0byB0aGUgLnR4dCIpCiAgICAgICAgc3lzLmV4aXQoMSkKCiAgICBuYXJyYXRpb24gPSBnZXRfbmFycmF0aW9uX3RleHQodHh0KQogICAgbmFycmF0aW9uX3Rva2VucyA9IHRva2VuaXplKG5hcnJhdGlvbikKICAgIG5hcnJhdGlvbl9ldmVudHMgPSBbCiAgICAgICAgeyJyYXciOiB0WyJyYXciXSwgIm5vcm0iOiBub3JtYWxpemVfd29yZCh0WyJyYXciXSl9CiAgICAgICAgZm9yIHQgaW4gbmFycmF0aW9uX3Rva2VucwogICAgXQoKICAgIHByaW50KGYiQXVkaW8gICAgICAgIDoge2F1ZGlvfSIpCiAgICBwcmludChmIk5hcnJhdGlvbiAgICA6IHtsZW4obmFycmF0aW9uX2V2ZW50cyl9IHdvcmRzIikKICAgIHByaW50KGYiUnVubmluZyBXaGlzcGVyIChiYXNlKSBvbiBDUFUuLi4iKQoKICAgIHdoaXNwZXJfZXZlbnRzID0gYXNyX3dvcmRzKGF1ZGlvKQogICAgcHJpbnQoZiJXaGlzcGVyICAgICAgOiB7bGVuKHdoaXNwZXJfZXZlbnRzKX0gZGV0ZWN0ZWQgd29yZHMiKQoKICAgIGFsaWduZWQgPSBhbGlnbihuYXJyYXRpb25fZXZlbnRzLCB3aGlzcGVyX2V2ZW50cykKCiAgICBhc3NfcGF0aCA9IG9zLnBhdGguam9pbihoZXJlLCBiYXNlICsgIi5hc3MiKQogICAgY29udGVudCA9IGJ1aWxkX2FzcyhhbGlnbmVkLCBmb250c2l6ZT1hcmdzLmZvbnRzaXplLAogICAgICAgICAgICAgICAgICAgICAgICBmb250bmFtZT1hcmdzLmZvbnRuYW1lLAogICAgICAgICAgICAgICAgICAgICAgICBtYXhfY2hhcnNfcGVyX2xpbmU9YXJncy5tYXhjaGFycywKICAgICAgICAgICAgICAgICAgICAgICAgbWluX2hvbGQ9YXJncy5taW5faG9sZCkKICAgIHdpdGggb3Blbihhc3NfcGF0aCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGYud3JpdGUoY29udGVudCkKICAgIHByaW50KGYiV3JvdGUgICAgICAgIDoge2Fzc19wYXRofSIpCgogICAgIyBhbHNvIGR1bXAgYSBKU09OIGZvciBkZWJ1Z2dpbmcKICAgIGZvciBhIGluIGFsaWduZWQ6CiAgICAgICAgYVsid29yZCJdID0gYS5wb3AoIndvcmQiKQogICAgd2l0aCBvcGVuKG9zLnBhdGguam9pbihoZXJlLCBiYXNlICsgIl90aW1pbmdzLmpzb24iKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChhbGlnbmVkLCBmLCBpbmRlbnQ9MSkKCiAgICAjIHByaW50IGZpcnN0IGZldyB3b3JkcyB0byBzYW5pdHkgY2hlY2sKICAgIHByaW50KCJcbkZpcnN0IDggdGltZWQgd29yZHM6IikKICAgIGZvciBhIGluIGFsaWduZWRbOjhdOgogICAgICAgIHByaW50KGYiICB7YVsnd29yZCddIXI6MjJzfSB7YVsnc3RhcnQnXTouM2Z9IC0+IHthWydlbmQnXTouM2Z9IikKICAgIHByaW50KGpzb24uZHVtcHMoeyJ0b3RhbF93b3JkcyI6IGxlbihhbGlnbmVkKSwgImR1cmF0aW9uX2VuZCI6IGFsaWduZWRbLTFdWyJlbmQiXX0sIGluZGVudD0xKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==').decode('utf-8'))
SRC.joinpath('enhance.py').write_text(
    base64.b64decode('IiIiQXVkaW8gcG9zdC1wcm9jZXNzaW5nIHBpcGVsaW5lIGZvciBDaGF0dGVyYm94IE5hbm8gb3V0cHV0LgoKU3RhZ2VzIChlYWNoIGluZGVwZW5kZW50bHkgc2tpcHBhYmxlIG9uIGZhaWx1cmUpOgogIDEuIExhdmFTUiAgIOKAlCBzcGVlY2ggZW5oYW5jZW1lbnQgKHdhcm10aCwgYmFuZHdpZHRoLCBjbGFyaXR5KQogIDIuIFJOTm9pc2UgIOKAlCBhcnRpZmFjdCByZW1vdmFsIChtZXRhbGxpYyBoaXNzLCBjbGlja3MpCiAgMy4gYXV0by1lZGl0b3Ig4oCUIHRyaW0gZGVhZCBzaWxlbmNlIGFuZCBzdHV0dGVycwogIDQuIEZGbXBlZyAgIOKAlCBtYXN0ZXJpbmcgKEVRLCBjb21wcmVzc2lvbiwgTFVGUyBub3JtYWxpemF0aW9uKQoiIiIKCmltcG9ydCBvcwppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKCl9MQVZBX01PREVMID0gTm9uZQpfTEFWQV9MT0NLID0gTm9uZQoKCmRlZiBfZ2V0X2xhdmEoKToKICAgICIiIkxhenkgc2luZ2xldG9uIGZvciBMYXZhU1IgbW9kZWwgKH41ME1CLCBsb2FkcyBvbmNlKS4iIiIKICAgIGdsb2JhbCBfTEFWQV9NT0RFTCwgX0xBVkFfTE9DSwogICAgaWYgX0xBVkFfTE9DSyBpcyBOb25lOgogICAgICAgIGltcG9ydCB0aHJlYWRpbmcKICAgICAgICBfTEFWQV9MT0NLID0gdGhyZWFkaW5nLkxvY2soKQogICAgd2l0aCBfTEFWQV9MT0NLOgogICAgICAgIGlmIF9MQVZBX01PREVMIGlzIE5vbmU6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGZyb20gTGF2YVNSLm1vZGVsIGltcG9ydCBMYXZhRW5oYW5jZTIKICAgICAgICAgICAgICAgIF9MQVZBX01PREVMID0gTGF2YUVuaGFuY2UyKCJZYXRoYXJ0aFMvTGF2YVNSIiwgImNwdSIpCiAgICAgICAgICAgICAgICBwcmludCgiW2VuaGFuY2VdIExhdmFTUiBtb2RlbCBsb2FkZWQiKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBwcmludChmIltlbmhhbmNlXSBMYXZhU1IgbG9hZCBmYWlsZWQ6IHtlfSIpCiAgICAgICAgICAgICAgICBfTEFWQV9NT0RFTCA9IEZhbHNlICAjIHNlbnRpbmVsOiBkbyBub3QgcmV0cnkKICAgIHJldHVybiBfTEFWQV9NT0RFTCBpZiBfTEFWQV9NT0RFTCBpcyBub3QgRmFsc2UgZWxzZSBOb25lCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBTdGFnZSAxOiBMYXZhU1Igc3BlZWNoIGVuaGFuY2VtZW50CiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBfc3RhZ2VfbGF2YShzcmMsIGRzdCk6CiAgICBsYXZhID0gX2dldF9sYXZhKCkKICAgIGlmIGxhdmEgaXMgTm9uZToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIGltcG9ydCBzb3VuZGZpbGUgYXMgc2YKICAgIGF1ZGlvLCBfc3IgPSBsYXZhLmxvYWRfYXVkaW8oc3RyKHNyYykpCiAgICBvdXQgPSBsYXZhLmVuaGFuY2UoYXVkaW8sIGRlbm9pc2U9RmFsc2UsIGJhdGNoPUZhbHNlKS5jcHUoKS5udW1weSgpLnNxdWVlemUoKQogICAgc2Yud3JpdGUoc3RyKGRzdCksIG91dCwgNDgwMDApCiAgICByZXR1cm4gVHJ1ZQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU3RhZ2UgMjogUk5Ob2lzZSBhcnRpZmFjdCByZW1vdmFsCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBfc3RhZ2Vfcm5ub2lzZShzcmMsIGRzdCk6CiAgICB0bXA0OCA9IHN0cihkc3QpICsgIi5fNDhrLndhdiIKICAgIHRyeToKICAgICAgICBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWyJmZm1wZWciLCAiLXkiLCAiLWkiLCBzdHIoc3JjKSwKICAgICAgICAgICAgICItYXIiLCAiNDgwMDAiLCAiLWFjIiwgIjEiLCAiLXNhbXBsZV9mbXQiLCAiczE2IiwgdG1wNDhdLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCBjaGVjaz1UcnVlLAogICAgICAgICkKICAgIGV4Y2VwdCBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCiAgICBkZW5vaXNlZCA9IHN0cihkc3QpICsgIi5fZG4ud2F2IgogICAgcmFuID0gRmFsc2UKICAgICMgdHJ5IENMSSBmaXJzdAogICAgdHJ5OgogICAgICAgIHN1YnByb2Nlc3MucnVuKFsiZGVub2lzZSIsIHRtcDQ4LCBkZW5vaXNlZF0sIGNhcHR1cmVfb3V0cHV0PVRydWUsIGNoZWNrPVRydWUpCiAgICAgICAgcmFuID0gVHJ1ZQogICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgIHBhc3MKICAgICMgZmFsbGJhY2s6IFB5dGhvbiBBUEkKICAgIGlmIG5vdCByYW46CiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcm9tIHB5cm5ub2lzZSBpbXBvcnQgUk5Ob2lzZQogICAgICAgICAgICBkZW5vaXNlciA9IFJOTm9pc2Uoc2FtcGxlX3JhdGU9NDgwMDApCiAgICAgICAgICAgIGZvciBfIGluIGRlbm9pc2VyLmRlbm9pc2Vfd2F2KHRtcDQ4LCBkZW5vaXNlZCk6CiAgICAgICAgICAgICAgICBwYXNzCiAgICAgICAgICAgIHJhbiA9IFRydWUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCiAgICAjIGNsZWFudXAgdGVtcCBpbnB1dHMKICAgIGZvciBmIGluIFt0bXA0OF06CiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoZik6CiAgICAgICAgICAgIG9zLnJlbW92ZShmKQogICAgaWYgbm90IHJhbiBvciBub3Qgb3MucGF0aC5leGlzdHMoZGVub2lzZWQpOgogICAgICAgIHJldHVybiBGYWxzZQoKICAgICMgcmVzdG9yZSBzYW1wbGUgcmF0ZQogICAgdHJ5OgogICAgICAgIHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbImZmbXBlZyIsICIteSIsICItaSIsIGRlbm9pc2VkLCAiLWFyIiwgIjI0MDAwIiwgc3RyKGRzdCldLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCBjaGVjaz1UcnVlLAogICAgICAgICkKICAgIGV4Y2VwdCBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcjoKICAgICAgICBzaHV0aWwuY29weTIoZGVub2lzZWQsIHN0cihkc3QpKQogICAgaWYgb3MucGF0aC5leGlzdHMoZGVub2lzZWQpOgogICAgICAgIG9zLnJlbW92ZShkZW5vaXNlZCkKICAgIHJldHVybiBUcnVlCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBTdGFnZSAzOiBhdXRvLWVkaXRvciBzaWxlbmNlIC8gYXJ0aWZhY3QgdHJpbW1pbmcKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIF9zdGFnZV9hdXRvZWRpdG9yKHNyYywgZHN0KToKICAgIHRyeToKICAgICAgICBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWyJhdXRvLWVkaXRvciIsIHN0cihzcmMpLCAiLS1vdXRwdXQiLCBzdHIoZHN0KSwKICAgICAgICAgICAgICItLXRocmVzaG9sZCIsICIwLjA0IiwgIi0tbWFyZ2luIiwgIjAuMiJdLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCBjaGVjaz1UcnVlLAogICAgICAgICkKICAgICAgICByZXR1cm4gZHN0LmV4aXN0cygpIGFuZCBkc3Quc3RhdCgpLnN0X3NpemUgPiAwCiAgICBleGNlcHQgKHN1YnByb2Nlc3MuQ2FsbGVkUHJvY2Vzc0Vycm9yLCBGaWxlTm90Rm91bmRFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBTdGFnZSA0OiBGRm1wZWcgbWFzdGVyaW5nIChFUSwgY29tcHJlc3Npb24sIExVRlMpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBfc3RhZ2VfbWFzdGVyKHNyYywgZHN0KToKICAgICMgcHJvYmUgc291cmNlIHNhbXBsZSByYXRlCiAgICB0cnk6CiAgICAgICAgcHJvYmUgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWyJmZnByb2JlIiwgIi12IiwgImVycm9yIiwgIi1zZWxlY3Rfc3RyZWFtcyIsICJhOjAiLAogICAgICAgICAgICAgIi1zaG93X2VudHJpZXMiLCAic3RyZWFtPXNhbXBsZV9yYXRlIiwgIi1vZiIsICJjc3Y9cD0wIiwKICAgICAgICAgICAgIHN0cihzcmMpXSwKICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCBjaGVjaz1UcnVlLAogICAgICAgICkKICAgICAgICBzciA9IGludChwcm9iZS5zdGRvdXQuc3RyaXAoKSkgb3IgMjQwMDAKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgc3IgPSAyNDAwMAoKICAgIGNoYWluID0gIiwiLmpvaW4oWwogICAgICAgICJoaWdocGFzcz1mPTgwIiwKICAgICAgICAiZXF1YWxpemVyPWY9MzAwMDp0PXE6dz0xOmc9LTIiLAogICAgICAgICJlcXVhbGl6ZXI9Zj0xNTA6dD1xOnc9MTpnPTEiLAogICAgICAgICJjb21wYW5kPWF0dGFja3M9MC4zOmRlY2F5cz0wLjg6cG9pbnRzPS04MC8tODB8LTIwLy0xNHwwLy03OmdhaW49MCIsCiAgICAgICAgImxvdWRub3JtPUk9LTE5OlRQPS0xLjU6TFJBPTExIiwKICAgIF0pCiAgICB0cnk6CiAgICAgICAgc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZmZtcGVnIiwgIi15IiwgIi1pIiwgc3RyKHNyYyksICItYWYiLCBjaGFpbiwKICAgICAgICAgICAgICItYXIiLCBzdHIoc3IpLCBzdHIoZHN0KV0sCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsIGNoZWNrPVRydWUsCiAgICAgICAgKQogICAgICAgIHJldHVybiBUcnVlCiAgICBleGNlcHQgc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3I6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBQdWJsaWMgQVBJCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tClNUQUdFUyA9IFsKICAgICgibGF2YSIsICAgICAgIF9zdGFnZV9sYXZhKSwKICAgICgicm5ub2lzZSIsICAgIF9zdGFnZV9ybm5vaXNlKSwKICAgICgiYXV0b2VkaXRvciIsIF9zdGFnZV9hdXRvZWRpdG9yKSwKICAgICgibWFzdGVyIiwgICAgIF9zdGFnZV9tYXN0ZXIpLApdCgoKZGVmIGVuaGFuY2VfYXVkaW8oaW5wdXRfcGF0aCwgb3V0cHV0X3BhdGg9Tm9uZSwgZW5hYmxlPVRydWUpOgogICAgIiIiUnVuIHRoZSBmdWxsIHBvc3QtcHJvY2Vzc2luZyBwaXBlbGluZS4KCiAgICBBcmdzOgogICAgICAgIGlucHV0X3BhdGg6ICBQYXRoIHRvIGlucHV0IFdBVi4KICAgICAgICBvdXRwdXRfcGF0aDogUGF0aCB0byBvdXRwdXQgV0FWIChkZWZhdWx0OiBvdmVyd3JpdGUgaW5wdXQpLgogICAgICAgIGVuYWJsZTogICAgICBGYWxzZSB0byBza2lwIGFsbCBwcm9jZXNzaW5nLgoKICAgIFJldHVybnM6CiAgICAgICAgKG91dHB1dF9wYXRoX3N0ciwgc3RhdHNfZGljdCkKICAgICIiIgogICAgaWYgbm90IGVuYWJsZToKICAgICAgICByZXR1cm4gc3RyKGlucHV0X3BhdGgpLCB7fQoKICAgIGlucHV0X3BhdGggPSBQYXRoKGlucHV0X3BhdGgpCiAgICBpZiBvdXRwdXRfcGF0aCBpcyBOb25lOgogICAgICAgIG91dHB1dF9wYXRoID0gaW5wdXRfcGF0aAogICAgZWxzZToKICAgICAgICBvdXRwdXRfcGF0aCA9IFBhdGgob3V0cHV0X3BhdGgpCgogICAgc3RhdHMgPSB7fQogICAgY3VycmVudCA9IGlucHV0X3BhdGgKCiAgICBmb3IgbmFtZSwgZm4gaW4gU1RBR0VTOgogICAgICAgIHRtcCA9IGlucHV0X3BhdGgucGFyZW50IC8gZiIudG1wX3tuYW1lfV97aW5wdXRfcGF0aC5uYW1lfSIKICAgICAgICB0cnk6CiAgICAgICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICAgICAgb2sgPSBmbihjdXJyZW50LCB0bXApCiAgICAgICAgICAgIGVsYXBzZWQgPSBmInt0aW1lLnRpbWUoKSAtIHQwOi4xZn1zIgogICAgICAgICAgICBpZiBvayBhbmQgdG1wLmV4aXN0cygpIGFuZCB0bXAuc3RhdCgpLnN0X3NpemUgPiAwOgogICAgICAgICAgICAgICAgY3VycmVudCA9IHRtcAogICAgICAgICAgICAgICAgc3RhdHNbbmFtZV0gPSBlbGFwc2VkCiAgICAgICAgICAgICAgICBwcmludChmIiAgW2VuaGFuY2VdIHtuYW1lfToge2VsYXBzZWR9IikKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHByaW50KGYiICBbZW5oYW5jZV0ge25hbWV9OiBza2lwcGVkIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHByaW50KGYiICBbZW5oYW5jZV0ge25hbWV9OiBmYWlsZWQgKHtlfSkiKQoKICAgICMgbW92ZSBmaW5hbCByZXN1bHQgdG8gb3V0cHV0X3BhdGgKICAgIGlmIGN1cnJlbnQgIT0gb3V0cHV0X3BhdGg6CiAgICAgICAgc2h1dGlsLm1vdmUoc3RyKGN1cnJlbnQpLCBzdHIob3V0cHV0X3BhdGgpKQoKICAgICMgY2xlYW51cCByZW1haW5pbmcgdGVtcCBmaWxlcwogICAgZm9yIG5hbWUsIF8gaW4gU1RBR0VTOgogICAgICAgIHRtcCA9IGlucHV0X3BhdGgucGFyZW50IC8gZiIudG1wX3tuYW1lfV97aW5wdXRfcGF0aC5uYW1lfSIKICAgICAgICBpZiB0bXAuZXhpc3RzKCkgYW5kIHRtcCAhPSBvdXRwdXRfcGF0aDoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgdG1wLnVubGluaygpCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICAgICAgcGFzcwoKICAgIHJldHVybiBzdHIob3V0cHV0X3BhdGgpLCBzdGF0cwo=').decode('utf-8'))
SRC.joinpath('reel_render.py').write_text(
    base64.b64decode('...').decode('utf-8'))
print('story_src installed:', sorted(p.name for p in SRC.glob('*.py')))


In [ ]:
# Cell 3 — Assets: fetch reels_batch.txt + 10 female voices + outro.
import os, json, subprocess, urllib.request, urllib.parse, shutil
from pathlib import Path

# Reels working dir #######################################################
REEL_ROOT = Path("/content/weekendpulse_reels")
V = REEL_ROOT / "voices"
O = REEL_ROOT / "output"
A = REEL_ROOT / "assets"
for d in (V, O, A):
    d.mkdir(parents=True, exist_ok=True)

REPO = "thechemiluminary/WeekendPulse"
RAW = "https://raw.githubusercontent.com"

# 1. Batch of approved reels (produced by the GitHub reel_batch workflow) ###
def fetch_raw(path):
    url = f"{RAW}/{REPO}/main/{path}"
    with urllib.request.urlopen(url, timeout=40) as r:
        return r.read()

try:
    batch_bytes = fetch_raw("reels_batch.txt")
    (REEL_ROOT / "reels_batch.txt").write_bytes(batch_bytes)
    batch = json.loads(batch_bytes.decode("utf-8"))
    print(f"Loaded reels_batch.txt: {len(batch['reels'])} reel(s)")
except Exception as e:
    print("WARN could not fetch a fresh reels_batch.txt:", e)
    batch = {"generated_at_utc": None, "count": 0, "reels": []}

# 2. Download 10 female voice folders + emotion clips ####################
def dl(url, dest):
    try:
        urllib.request.urlretrieve(url, dest)
    except Exception:
        return False
    return Path(dest).exists() and Path(dest).stat().st_size > 1000

# voices-emotion/<voice>/<emotion>.flac  (OwenTyme/voice-zero, public)
VOICE_REPO = "OwenTyme/voice-zero"
VOICE_PATH = "voices-emotion"
FEMALE_BASE = [
    "kristin_hughes", "jodi_krangle", "karen_savage",
    "emily_cripps", "cori_samuel", "mil_nicholson",
    "amy_koenig", "alana_jordan", "emily_anderson", "anna_simon",
]
# only fetch the emotions this project uses; neutral always
EMOTION_CLIPS = ["neutral", "excited", "surprised"]
got_voices = []
for v in FEMALE_BASE:
    vdir = V / v
    vdir.mkdir(parents=True, exist_ok=True)
    have = 0
    for em in EMOTION_CLIPS:
        dest = vdir / f"{em}.flac"
        url = (f"{RAW}/{VOICE_REPO}/main/{VOICE_PATH}/"
               f"{urllib.parse.quote(v)}/{em}.flac")
        if dl(url, dest):
            have += 1
    if have:
        got_voices.append(v)
        print(f"  voice + {v} ({have}/{len(EMOTION_CLIPS)} clips)")
    else:
        print(f"  MISSING voice {v}")
print(f"Downloaded {len(got_voices)} female voice(s) into {V}: {got_voices}")
if not got_voices:
    raise RuntimeError("No voices downloaded - check VOICE_REPO/path above")

# 3. Outro clip (user's fixed 4s outro) ###################################
outro_src = A / "outro.mp4"
outro_url = f"{RAW}/{REPO}/main/outro/{'outro.mp4'}"
if not outro_src.exists() or outro_src.stat().st_size < 1000:
    dl(outro_url, outro_src)
if outro_src.exists() and outro_src.stat().st_size > 1000:
    print("outro.mp4 ready:", outro_src.stat().st_size // 1024, "KiB")
else:
    print("WARN outro.mp4 missing — reels will render without the outro clip")

print("\nAssets ready. Review reels_batch.txt before rendering.")


In [ ]:
# Cell 4 — RENDER all reels in the batch (one MP4 per reel).
import json, time
from pathlib import Path

REEL_ROOT = Path("/content/weekendpulse_reels")
OUT = REEL_ROOT / "output"

# Load batch built in the Assets cell
with open(REEL_ROOT / "reels_batch.txt", encoding="utf-8") as f:
    batch = json.load(f)

entries = batch.get("reels", [])
entries = entries[:6]                       # soft cap REEL_MAX_PER_RUN=6
print(f"Rendering {len(entries)} reel(s) ...\n")

# the renderer module (written to disk by the 'write files' cell)
import sys
sys.path.insert(0, str(REEL_ROOT / "story_src"))
import reel_render

t_start = time.time()
results = reel_render.render_all(entries)
print("\n=== SUMMARY ===")
for r in results:
    print(f"  {r['slug']:<30} {r['duration_s']:>5.1f}s  emotion={r['emotion']:<9} music={r['music']}")

# persist summaries for the preview cell
(REEL_ROOT / "rendered_results.json").write_text(
    json.dumps(results, indent=2), encoding="utf-8")
(REEL_ROOT / "rendered_list.txt").write_text(
    "\n".join(str(OUT / r["slug"]) + ".mp4" for r in results), encoding="utf-8")
print(f"\nTotal render time: {time.time() - t_start:.0f}s")
print("Outputs:", OUT)


In [ ]:
# Cell 5 — Preview each rendered reel, switching with a slider (Next/Previous).
# Uses ipywidgets + embedded IPython.display.Video so it plays reliably in Colab.
import json
from pathlib import Path
from IPython.display import display, Video

REEL_ROOT = Path("/content/weekendpulse_reels")
OUT = REEL_ROOT / "output"

# --- order rendered mp4s by slug (render summary order if present)
results = []
res_path = REEL_ROOT / "rendered_results.json"
if res_path.exists():
    with open(res_path, encoding="utf-8") as f:
        results = json.load(f)
paths = [p for r in results
         if (p := OUT / f"{r['slug']}.mp4").exists()]
if not paths:
    paths = sorted(OUT.glob("*.mp4"))

if not paths:
    print("No rendered MP4s yet — run the RENDER cell first.")
else:
    def show(i):
        i = max(0, min(len(paths) - 1, i))
        path = paths[i]
        meta = results[i] if i < len(results) else {}
        print(f"{i+1}/{len(paths)}  {path.name}  "
              f"emotion={meta.get('emotion','?')}  music={meta.get('music','-')}  "
              f"{meta.get('duration_s',0):.1f}s")
        display(Video(filename=str(path), embed=True,
                      metadata={"mimetype": "video/mp4"}, width=360))
        state = REEL_ROOT / "preview_state.json"
        state.write_text(json.dumps({"i": i}), encoding="utf-8")

    try:
        import ipywidgets as widgets
        idx = 0
        s = REEL_ROOT / "preview_state.json"
        if s.exists():
            idx = json.loads(s.read_text(encoding="utf-8")).get("i", 0)
        slider = widgets.IntSlider(min=0, max=len(paths) - 1, value=idx,
                                   description="Reel")
        out = widgets.Output()
        # re-draw on slider change
        def _on_change(change):
            out.clear_output(wait=True)
            with out:
                show(change["new"])
        slider.observe(_on_change, names="value")
        display(widgets.VBox([
            widgets.HBox([widgets.Label("Index:"), slider]),
            out,
        ]))
        with out:
            show(slider.value)
    except Exception as _e:
        # fallback: static preview of the first reel
        print("ipywidgets unavailable — showing first reel.")
        show(0)


In [ ]:
# Cell 6 — Download rendered MP4s (zip them all, plus individual links).
import zipfile
from pathlib import Path
from IPython.display import FileLink, HTML, display

OUT = Path("/content/weekendpulse_reels/output")
mp4s = sorted(str(p) for p in OUT.glob("*.mp4"))
if not mp4s:
    print("No rendered MP4s yet — run the RENDER cell first.")
else:
    zip_path = OUT / "reels.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for m in mp4s:
            z.write(m, arcname=Path(m).name)
    print(f"{len(mp4s)} reel(s) in {zip_path.name}  ({zip_path.stat().st_size//1024} KiB)")
    display(FileLink(str(zip_path), result_html_prefix="Download all: "))
    display(HTML("<h4>Download individually</h4>"))
    for m in mp4s:
        display(FileLink(m))
